<a href="https://colab.research.google.com/github/varshitha-1821/AV.SC.U4AIE23116/blob/main/notebooks/unit2/unit2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###🎮 Environments:

- [FrozenLake-v1](https://gymnasium.farama.org/environments/toy_text/frozen_lake/)
- [Taxi-v3](https://gymnasium.farama.org/environments/toy_text/taxi/)

###📚 RL-Library:

- Python and NumPy
- [Gymnasium](https://gymnasium.farama.org/)

We're constantly trying to improve our tutorials, so **if you find some issues in this notebook**, please [open an issue on the GitHub Repo](https://github.com/huggingface/deep-rl-class/issues).

## Objectives of this notebook 🏆

At the end of the notebook, you will:

- Be able to use **Gymnasium**, the environment library.
- Be able to code a Q-Learning agent from scratch.
- Be able to **push your trained agent and the code to the Hub** with a nice video replay and an evaluation score 🔥.




In [ ]:
!pip install -r https://raw.githubusercontent.com/huggingface/deep-rl-class/main/notebooks/unit2/requirements-unit2.txt

In [ ]:
!sudo apt-get update
!sudo apt-get install -y python3-opengl
!apt install ffmpeg xvfb
!pip3 install pyvirtualdisplay

In [ ]:
#import os
#os.kill(os.getpid(), 9)

In [ ]:
# Virtual display
from pyvirtualdisplay import Display

virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()

In [ ]:
import numpy as np
import gymnasium as gym
import random
import imageio
import os
import tqdm

import pickle
from tqdm.notebook import tqdm

In [ ]:
# Create the FrozenLake-v1 environment using 4x4 map and non-slippery version and render_mode="rgb_array"
# ---------- Part 1: FrozenLake (4x4, non-slippery) ----------

# 1) Create the game. rgb_array lets us record a replay video later.
env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=False, render_mode="rgb_array")

# 2) The Q-table: one row per state (16 squares), one column per action (4 moves), all zeros to start.
state_space = env.observation_space.n
action_space = env.action_space.n
Qtable_frozenlake = np.zeros((state_space, action_space))
print("States:", state_space, "| Actions:", action_space)

# 3) Greedy policy: in a state, pick the action with the highest Q-value (exploit what we know).
def greedy_policy(Qtable, state):
    return np.argmax(Qtable[state][:])

# 4) Epsilon-greedy policy: usually exploit, but with probability epsilon try a random action (explore).
def epsilon_greedy_policy(Qtable, state, epsilon):
    if random.uniform(0, 1) > epsilon:
        return greedy_policy(Qtable, state)
    return env.action_space.sample()

# 5) Hyperparameters (the course's values)
n_training_episodes = 10000   # how many games to play while learning
learning_rate = 0.7           # how fast Q-values move toward new information
n_eval_episodes = 100         # test games after training
env_id = "FrozenLake-v1"
max_steps = 99                # max moves per game
gamma = 0.95                  # how much future rewards matter
eval_seed = []                # empty = random starts (only Taxi uses fixed seeds)
max_epsilon = 1.0             # start: 100% exploring
min_epsilon = 0.05            # end: 5% exploring
decay_rate = 0.0005           # how fast exploring shrinks

# 6) Training loop: play many games and update the Q-table after every move.
def train(n_training_episodes, min_epsilon, max_epsilon, decay_rate, env, max_steps, Qtable):
    for episode in tqdm(range(n_training_episodes)):
        # explore less and less as training goes on
        epsilon = min_epsilon + (max_epsilon - min_epsilon) * np.exp(-decay_rate * episode)
        state, info = env.reset()
        for step in range(max_steps):
            action = epsilon_greedy_policy(Qtable, state, epsilon)
            new_state, reward, terminated, truncated, info = env.step(action)
            # Q-learning update: Q(s,a) += lr * (reward + gamma * best_next_Q - Q(s,a))
            Qtable[state][action] = Qtable[state][action] + learning_rate * (
                reward + gamma * np.max(Qtable[new_state]) - Qtable[state][action]
            )
            if terminated or truncated:
                break
            state = new_state
    return Qtable

# 7) Evaluation (provided by the course): play test games using only the greedy policy.
def evaluate_agent(env, max_steps, n_eval_episodes, Q, seed):
    episode_rewards = []
    for episode in tqdm(range(n_eval_episodes)):
        if seed:
            state, info = env.reset(seed=seed[episode])
        else:
            state, info = env.reset()
        total_rewards_ep = 0
        for step in range(max_steps):
            action = greedy_policy(Q, state)
            new_state, reward, terminated, truncated, info = env.step(action)
            total_rewards_ep += reward
            if terminated or truncated:
                break
            state = new_state
        episode_rewards.append(total_rewards_ep)
    return np.mean(episode_rewards), np.std(episode_rewards)

# 8) Train, then evaluate.
Qtable_frozenlake = train(n_training_episodes, min_epsilon, max_epsilon, decay_rate, env, max_steps, Qtable_frozenlake)
mean_reward, std_reward = evaluate_agent(env, max_steps, n_eval_episodes, Qtable_frozenlake, eval_seed)
print(f"Mean_reward={mean_reward:.2f} +/- {std_reward:.2f}")

In [ ]:
# Older Gymnasium that still includes Taxi-v3 (the checker only recognises v3)
!pip install -q "gymnasium==0.29.1"

In [ ]:
# ---------- Part 2: Taxi-v3 ----------

# 1) Create the Taxi game (500 states, 6 actions: move x4, pick up, drop off)
env = gym.make("Taxi-v3", render_mode="rgb_array")
state_space = env.observation_space.n
action_space = env.action_space.n
Qtable_taxi = np.zeros((state_space, action_space))   # 500 x 6 table of zeros
print("Q-table shape:", Qtable_taxi.shape)

# 2) Hyperparameters (Taxi values)
n_training_episodes = 25000
learning_rate = 0.7
n_eval_episodes = 100

# DO NOT MODIFY: fixed starting positions, so everyone is tested on the same 100 games
eval_seed = [
    16, 54, 165, 177, 191, 191, 120, 80, 149, 178, 48, 38, 6, 125, 174, 73, 50, 172, 100, 148,
    146, 6, 25, 40, 68, 148, 49, 167, 9, 97, 164, 176, 61, 7, 54, 55, 161, 131, 184, 51,
    170, 12, 120, 113, 95, 126, 51, 98, 36, 135, 54, 82, 45, 95, 89, 59, 95, 124, 9, 113,
    58, 85, 51, 134, 121, 169, 105, 21, 30, 11, 50, 65, 12, 43, 82, 145, 152, 97, 106, 55,
    31, 85, 38, 112, 102, 168, 123, 97, 21, 83, 158, 26, 80, 63, 5, 81, 32, 11, 28, 148,
]

env_id = "Taxi-v3"
max_steps = 99
gamma = 0.95
max_epsilon = 1.0
min_epsilon = 0.05
decay_rate = 0.005     # Taxi decays exploration faster than FrozenLake

# 3) Train with the same train() function as before
Qtable_taxi = train(n_training_episodes, min_epsilon, max_epsilon, decay_rate, env, max_steps, Qtable_taxi)

# 4) Evaluate on the fixed seeds. This is the number the certification uses.
mean_reward, std_reward = evaluate_agent(env, max_steps, n_eval_episodes, Qtable_taxi, eval_seed)
print(f"Mean_reward={mean_reward:.2f} +/- {std_reward:.2f}")
print(f"Leaderboard score (mean - std) = {mean_reward - std_reward:.2f}")

In [ ]:
from getpass import getpass
from huggingface_hub import login

# Paste your hf_... write token when the box appears (it stays hidden as you type)
login(token=getpass("Enter your token key here: "))

In [ ]:
# ---- Course helper code (unchanged): defines record_video() and push_to_hub() ----
from huggingface_hub import HfApi, snapshot_download
from huggingface_hub.repocard import metadata_eval_result, metadata_save

from pathlib import Path
import datetime
import json


def record_video(env, Qtable, out_directory, fps=1):
    images = []
    terminated = False
    truncated = False
    state, info = env.reset(seed=random.randint(0, 500))
    img = env.render()
    images.append(img)
    while not terminated or truncated:
        action = np.argmax(Qtable[state][:])
        state, reward, terminated, truncated, info = env.step(action)
        img = env.render()
        images.append(img)
    imageio.mimsave(out_directory, [np.array(img) for i, img in enumerate(images)], fps=fps)


def push_to_hub(repo_id, model, env, video_fps=1, local_repo_path="hub"):
    _, repo_name = repo_id.split("/")
    eval_env = env
    api = HfApi()

    # Step 1: Create the repo
    repo_url = api.create_repo(repo_id=repo_id, exist_ok=True)

    # Step 2: Download files
    repo_local_path = Path(snapshot_download(repo_id=repo_id))

    # Step 3: Save the model
    if env.spec.kwargs.get("map_name"):
        model["map_name"] = env.spec.kwargs.get("map_name")
        if env.spec.kwargs.get("is_slippery", "") == False:
            model["slippery"] = False

    with open((repo_local_path) / "q-learning.pkl", "wb") as f:
        pickle.dump(model, f)

    # Step 4: Evaluate the model and save the results
    mean_reward, std_reward = evaluate_agent(
        eval_env, model["max_steps"], model["n_eval_episodes"], model["qtable"], model["eval_seed"]
    )
    evaluate_data = {
        "env_id": model["env_id"],
        "mean_reward": mean_reward,
        "n_eval_episodes": model["n_eval_episodes"],
        "eval_datetime": datetime.datetime.now().isoformat(),
    }
    with open(repo_local_path / "results.json", "w") as outfile:
        json.dump(evaluate_data, outfile)

    # Step 5: Create the model card
    env_name = model["env_id"]
    if env.spec.kwargs.get("map_name"):
        env_name += "-" + env.spec.kwargs.get("map_name")
    if env.spec.kwargs.get("is_slippery", "") == False:
        env_name += "-" + "no_slippery"

    metadata = {}
    metadata["tags"] = [env_name, "q-learning", "reinforcement-learning", "custom-implementation"]

    eval = metadata_eval_result(
        model_pretty_name=repo_name,
        task_pretty_name="reinforcement-learning",
        task_id="reinforcement-learning",
        metrics_pretty_name="mean_reward",
        metrics_id="mean_reward",
        metrics_value=f"{mean_reward:.2f} +/- {std_reward:.2f}",
        dataset_pretty_name=env_name,
        dataset_id=env_name,
    )
    metadata = {**metadata, **eval}

    model_card = f"""
  # **Q-Learning** Agent playing1 **{model["env_id"]}**
  This is a trained model of a **Q-Learning** agent playing **{model["env_id"]}** .

  ## Usage

  model = load_from_hub(repo_id="{repo_id}", filename="q-learning.pkl")

  # Don't forget to check if you need to add additional attributes (is_slippery=False etc)
  env = gym.make(model["env_id"])
  """

    readme_path = repo_local_path / "README.md"
    readme = ""
    if readme_path.exists():
        with readme_path.open("r", encoding="utf8") as f:
            readme = f.read()
    else:
        readme = model_card
    with readme_path.open("w", encoding="utf-8") as f:
        f.write(readme)

    metadata_save(readme_path, metadata)

    # Step 6: Record a video
    video_path = repo_local_path / "replay.mp4"
    record_video(env, model["qtable"], video_path, video_fps)

    # Step 7: Push everything to the Hub
    api.upload_folder(repo_id=repo_id, folder_path=repo_local_path, path_in_repo=".")
    print("Your model is pushed to the Hub. You can view your model here: ", repo_url)

In [ ]:
model = {
    "env_id": env_id,
    "max_steps": max_steps,
    "n_training_episodes": n_training_episodes,
    "n_eval_episodes": n_eval_episodes,
    "eval_seed": eval_seed,
    "learning_rate": learning_rate,
    "gamma": gamma,
    "max_epsilon": max_epsilon,
    "min_epsilon": min_epsilon,
    "decay_rate": decay_rate,
    "qtable": Qtable_taxi,
}

username = "Yoga-Varshitha-Jajula"
repo_name = "q-Taxi-v3"

push_to_hub(repo_id=f"{username}/{repo_name}", model=model, env=env)